In [20]:
import pandas as pd
import polars as pl
import numpy as np
import os
from tqdm.auto import tqdm
from matplotlib import pyplot as plt
import pickle

from sklearn.metrics import r2_score
from sklearn.ensemble import VotingRegressor

import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None

In [21]:
TRAINING = True
feature_names = [f"feature_{i:02d}" for i in range(79)] + [f"responder_{idx}_lag_1" for idx in range(9)]
label_name = 'responder_6'
weight_name = 'weight'

df = pl.scan_parquet("train.parquet").collect().to_pandas()
valid = pl.scan_parquet("valid.parquet").collect().to_pandas()
df.shape, valid.shape

((21022056, 104), (1082224, 104))

In [22]:
df = pd.concat([df, valid]).reset_index(drop=True) # Combine train and valid

In [23]:
X_train = df[ feature_names ]
y_train = df[ label_name ]
w_train = df[ "weight" ]
X_valid = valid[ feature_names ]
y_valid = valid[ label_name ]
w_valid = valid[ "weight" ]

X_train.shape, y_train.shape, w_train.shape, X_valid.shape, y_valid.shape, w_valid.shape

((22104280, 88),
 (22104280,),
 (22104280,),
 (1082224, 88),
 (1082224,),
 (1082224,))

In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [25]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(1)  # [max_len, 1, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [sequence_len, batch_size, d_model]
        seq_len = x.size(0)
        x = x + self.pe[:seq_len, :]
        return x


In [26]:
import torch.nn.functional as F
import pytorch_lightning as pl
from torch.optim.lr_scheduler import ReduceLROnPlateau

class TransformerRegressor(pl.LightningModule):
    def __init__(
        self,
        input_dim: int,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 2,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
        lr: float = 1e-3,
        weight_decay: float = 1e-4
    ):
        """
        Args:
            input_dim: number of features (each feature is treated as one “token”)
            d_model: size of the transformer embeddings
            nhead: number of heads in multi-head attention
            num_layers: number of transformer encoder layers
            dim_feedforward: hidden dim in the feed-forward network within each encoder layer
            dropout: dropout probability
            lr: learning rate
            weight_decay: weight decay
        """
        super().__init__()
        self.save_hyperparameters()

        # We will treat each feature as a token.
        # So, the input is [batch_size, input_dim].
        # We embed it to [batch_size, input_dim, d_model].
        self.feature_embedding = nn.Linear(1, d_model)

        # Positional Encoding (for each feature index as position)
        self.pos_encoder = PositionalEncoding(d_model)

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # Final regressor head
        self.regressor = nn.Linear(d_model, 1)

        self.lr = lr
        self.weight_decay = weight_decay
        self.validation_step_outputs = []

    def forward(self, x):
        """
        x shape: [batch_size, input_dim]
        We want shape: [seq_len, batch_size, d_model] for the Transformer.
        We'll treat each feature as one token, so seq_len = input_dim.
        """

        # 1) Reshape x to [batch_size, input_dim, 1]
        x = x.unsqueeze(-1)   # (B, input_dim, 1)

        # 2) Embed each scalar feature -> embedding dimension d_model
        #    result: (B, input_dim, d_model)
        x = self.feature_embedding(x)

        # 3) PyTorch transformer expects shape: [seq_len, batch_size, d_model]
        x = x.permute(1, 0, 2)  # (input_dim, B, d_model)

        # 4) Add positional encoding
        x = self.pos_encoder(x)  # (input_dim, B, d_model)

        # 5) Forward pass through Transformer encoder
        x = self.transformer_encoder(x)  # (input_dim, B, d_model)

        # 6) Average pool across “tokens” (the features)
        x = x.mean(dim=0)  # (B, d_model)

        # 7) Final linear -> output shape: (B, 1)
        x = self.regressor(x)

        # Multiply by 5 to match your existing scale approach
        return 5 * x.squeeze(-1)  # (B,)

    def training_step(self, batch, batch_idx):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w
        loss = loss.mean()
        self.log('train_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        return loss

    def validation_step(self, batch, batch_idx):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w
        loss = loss.mean()
        self.log('val_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        self.validation_step_outputs.append((y_hat, y, w))
        return loss

    def on_validation_epoch_end(self):
        y = torch.cat([item[1] for item in self.validation_step_outputs]).cpu().numpy()
        y_hat = torch.cat([item[0] for item in self.validation_step_outputs]).cpu().numpy()
        weights = torch.cat([item[2] for item in self.validation_step_outputs]).cpu().numpy()

        # R2 metric
        val_r_square = 1 - (
            np.average((y_hat - y) ** 2, weights=weights) 
            / (np.average((y) ** 2, weights=weights) + 1e-38)
        )
        self.log("val_r_square", val_r_square, prog_bar=True, on_step=False, on_epoch=True)

        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',
            }
        }

    def on_train_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        epoch = self.trainer.current_epoch
        metrics = {k: v.item() if isinstance(v, torch.Tensor) else v for k, v in self.trainer.logged_metrics.items()}
        formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}
        print(f"Epoch {epoch}: {formatted_metrics}")


In [ ]:
import os
import pickle
import gc
import polars
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, Timer
from pytorch_lightning.loggers import WandbLogger
from torch.utils.data import Dataset, DataLoader


TRAINING = True


class custom_args():
    def __init__(self):
        self.usegpu = True
        self.gpuid = 0
        self.seed = 42
        self.model = 'transformer'
        self.use_wandb = False
        self.project = 'js-xs-transformer'
        self.dname = "./input_df/"
        self.loader_workers = 4
        self.bs = 8192
        self.lr = 1e-3
        self.weight_decay = 1e-4
        self.dropouts = [0.2, 0.1, 0.1]  # not used in the Transformer example
        self.n_hidden = [512, 512, 512, 256]
        self.patience = 25
        self.max_epochs = 2500
        self.N_fold = 5

my_args = custom_args()

class CustomDataset(Dataset):
    def __init__(self, df, accelerator):
        self.features = torch.FloatTensor(df[feature_names].values).to(accelerator)
        self.labels = torch.FloatTensor(df[label_name].values).to(accelerator)
        self.weights = torch.FloatTensor(df[weight_name].values).to(accelerator)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        w = self.weights[idx]
        return x, y, w

class DataModule(pl.LightningDataModule):
    def __init__(self, train_df, batch_size, valid_df=None, accelerator='cpu'):
        super().__init__()
        self.df = train_df
        self.batch_size = batch_size
        self.dates = self.df['date_id'].unique()
        self.accelerator = accelerator
        self.train_dataset = None
        self.valid_df = valid_df
        self.val_dataset = None

    def setup(self, fold=0, N_fold=5, stage=None):
        selected_dates = [date for ii, date in enumerate(self.dates) if ii % N_fold != fold]
        df_train = self.df.loc[self.df['date_id'].isin(selected_dates)]
        self.train_dataset = CustomDataset(df_train, self.accelerator)
        if self.valid_df is not None:
            df_valid = self.valid_df
            self.val_dataset = CustomDataset(df_valid, self.accelerator)

    def train_dataloader(self, n_workers=0):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=n_workers)

    def val_dataloader(self, n_workers=0):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=n_workers)


# ========== 2) Import the TransformerRegressor from above ==========

# (Paste the TransformerRegressor and PositionalEncoding classes here, or import them from a separate file.)

pl.seed_everything(my_args.seed)

device = torch.device(f'cuda:{my_args.gpuid}' if torch.cuda.is_available() and my_args.usegpu else 'cpu')
accelerator = 'gpu' if torch.cuda.is_available() and my_args.usegpu else 'cpu'
loader_device = 'cpu'

data_module = DataModule(df, batch_size=my_args.bs, valid_df=valid, accelerator=loader_device)
gc.collect()

# ========== 3) Training loop with k-fold cross-validation ==========

for fold in range(my_args.N_fold):
    data_module.setup(fold, my_args.N_fold)

    input_dim = data_module.train_dataset.features.shape[1]
    
    # Instantiate the Transformer
    model = TransformerRegressor(
        input_dim=input_dim,
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1,
        lr=my_args.lr,
        weight_decay=my_args.weight_decay
    )

    # Optionally use W&B
    if my_args.use_wandb:
        wandb_run = wandb.init(project=my_args.project, config=vars(my_args), reinit=True)
        logger = WandbLogger(experiment=wandb_run)
    else:
        logger = None

    early_stopping = EarlyStopping('val_loss', patience=my_args.patience, mode='min', verbose=False)
    checkpoint_callback = ModelCheckpoint(
        monitor='val_loss',
        mode='min',
        save_top_k=1,
        verbose=False,
        dirpath='notebook_logs/',
        filename=f"./models/transformer_{fold}.model"
    )
    timer = Timer()

    trainer = Trainer(
        max_epochs=my_args.max_epochs,
        accelerator=accelerator,
        devices=[my_args.gpuid] if my_args.usegpu else None,
        logger=logger,
        callbacks=[early_stopping, checkpoint_callback, timer],
        enable_progress_bar=True
    )

    trainer.fit(
        model, 
        data_module.train_dataloader(my_args.loader_workers), 
        data_module.val_dataloader(my_args.loader_workers)
    )
    print(f'Fold-{fold} Training completed in {timer.time_elapsed("train"):.2f}s')


Seed set to 42


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX A6000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name                | Type               | Params  | Mode 
--------------------------------------------------------------------
0 | feature_embedding   | Linear             | 128     | train
1 | pos_encoder         | PositionalEncoding | 0       | train
2 | transformer_encoder | TransformerEncoder | 100.0 K | train
3 | regressor           | Linear             | 65      | train
--------------------------------------------------------------------
100 K     Trainable param

Epoch 0: 100%|██████████| 2159/2159 [09:15<00:00,  3.89it/s, v_num=0, val_r_square=nan.0]
Fold-0 Training completed in 555.21s


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name                | Type               | Params  | Mode 
--------------------------------------------------------------------
0 | feature_embedding   | Linear             | 128     | train
1 | pos_encoder         | PositionalEncoding | 0       | train
2 | transformer_encoder | TransformerEncoder | 100.0 K | train
3 | regressor           | Linear             | 65      | train
--------------------------------------------------------------------
100 K     Trainable params
0         Non-trainable params
100 K     Total params
0.401     Total estimated model params size (MB)
25        Modules in train mode
0         Modules in eval mode


Epoch 0:  53%|█████▎    | 1134/2158 [04:50<04:22,  3.90it/s, v_num=1]      